In [1]:
import xarray as xr
import pandas as pd
import numpy as np
from pathlib import Path
import datetime
from src.get_rda_era5 import ERA5DataSource
import re
import zarr

In [2]:
data = xr.open_zarr("data/raw_data/clustering_glade.zarr")

In [3]:
days = data["day"].values
_, first_idx = np.unique(days, return_index=True)

# keep first occurrence only
data = data.isel(day=np.sort(first_idx))

data = data.sel(day=slice(None, np.datetime64("2019-12-31")))

In [4]:
data.load()

<xarray.Dataset> Size: 8GB
Dimensions:    (day: 3414, channel: 10, latitude: 190, longitude: 304)
Coordinates:
  * channel    (channel) <U4 160B 'z850' 'z500' 'q850' ... 'v500' 'w850' 'w500'
  * day        (day) datetime64[ns] 27kB 2002-04-02 2002-04-03 ... 2019-12-22
  * latitude   (latitude) float64 2kB 60.5 60.25 60.0 59.75 ... 13.75 13.5 13.25
  * longitude  (longitude) float64 2kB 227.5 227.8 228.0 ... 302.8 303.0 303.2
Data variables:
    Z          (day, channel, latitude, longitude) float32 8GB 1.496e+04 ... ...

In [5]:
outlooks = xr.open_dataset('data/raw_data/grid_outlooks.nc')
missing_dates = [
    '200204250000', '200208300000', '200304150000', '200304160000',
    '200306250000', '200307270000', '200307280000', '200312280000',
    '200404140000', '200408090000', '200905280000', '201105210000',
    '202005240000', '200510240000'
]
missing_dt = pd.to_datetime(missing_dates, format='%Y%m%d%H%M')
# selection of desired days
pph = xr.open_dataset('data/raw_data/labelled_pph.nc')
pph_time = pd.to_datetime(pph['time'].values, format='%Y%m%d%H%M')

cats = ['SLGT', 'ENH', 'MDT', 'HIGH']
mask = np.isin(pph['MAX_CAT'].values, cats) & (pph_time> '2002-04-01') & (pph_time < '2020-01-01') & ~np.isin(pph_time, missing_dt)

times = pph_time[mask]
times_shifted = times + np.timedelta64(1, 'D')

grid_outlooks = outlooks.sel(time = outlooks['time'] >='200203300000')
# finding x and y to make center for each date
grouped = grid_outlooks['prob'].groupby('time')

# Step 1: Find all points with the maximum prob for each day and compute mean coordinates
def find_mean_coords(group):
    max_prob = group.max()  # Maximum value in the group
    if max_prob == 0:
        mean_x = group['x'].mean().item()
        mean_y = group['y'].mean().item()
    else:
        # Select all points with prob == max_prob
        max_points = group.where(group == max_prob, drop=True)
        # Compute the mean of x and y
        mean_x = max_points['x'].mean().item()
        mean_y = max_points['y'].mean().item()
    return xr.Dataset({'nearest_x': mean_x, 'nearest_y': mean_y})

# Apply the function to each group
center_coords = grouped.map(find_mean_coords)

/glade/derecho/scratch/milesep/tmp/ipykernel_87741/2990620472.py:38: UserWarning: The `squeeze` kwarg to GroupBy is being removed.Pass .groupby(..., squeeze=False) to disable squeezing, which is the new default, and to silence this warning.
  center_coords = grouped.map(find_mean_coords)


In [6]:
pph = pph.assign_coords(
    time=pd.to_datetime(pph.time.values, format="%Y%m%d%H%M")
)

center_coords = center_coords.assign_coords(
    time=pd.to_datetime(center_coords.time.values, format="%Y%m%d%H%M")
)

cc = center_coords.sel(time=times)

latlon = xr.Dataset(
    {
        "lat": pph.lat,
        "lon": pph.lon,
    }
)

interp_latlon = latlon.interp(
    x=xr.DataArray(cc.nearest_x, dims="time"),
    y=xr.DataArray(cc.nearest_y, dims="time"),
    method="linear"
)

In [7]:
STEP = 0.25
HALF_DEG = 12.0
HALF_N = int(HALF_DEG / STEP)  # 48
N = 2 * HALF_N + 1             # 97

def extract_window(era_day, lat, lon):
    w = era_day.sel(
        latitude=slice(lat + HALF_DEG, lat - HALF_DEG),
        longitude=slice(360 + lon - HALF_DEG, 360 + lon + HALF_DEG),
    )

    w = w.rename(latitude="y", longitude="x")
    w = w.assign_coords(
        y=np.arange(N),
        x=np.arange(N),
    )
    return w

In [8]:
out = []

for d in data.day.values:
    lat = round(interp_latlon.sel(time = d).lat.item() * 4)/4
    lon = round(interp_latlon.sel(time = d).lon.item() * 4)/4

    era_day = data.sel(day=d)
    w = extract_window(era_day, lat, lon)
    w = w.expand_dims(day=[d])
    out.append(w)



In [9]:
window_ds = xr.concat(out, dim="day")
window_ds

<xarray.Dataset> Size: 1GB
Dimensions:  (day: 3414, channel: 10, y: 97, x: 97)
Coordinates:
  * day      (day) datetime64[ns] 27kB 2002-04-02 2002-04-03 ... 2019-12-22
  * channel  (channel) <U4 160B 'z850' 'z500' 'q850' ... 'v500' 'w850' 'w500'
  * y        (y) int64 776B 0 1 2 3 4 5 6 7 8 9 ... 88 89 90 91 92 93 94 95 96
  * x        (x) int64 776B 0 1 2 3 4 5 6 7 8 9 ... 88 89 90 91 92 93 94 95 96
Data variables:
    Z        (day, channel, y, x) float32 1GB 1.5e+04 1.498e+04 ... 0.08615

In [10]:
window_ds.to_netcdf('data/raw_data/clustering_fields.nc')